In [ ]:
import os
import random
import time
from itertools import chain
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor
from torch_geometric.data import Data
from tqdm import tqdm

import barostat_parameters
from barostat_utils import (
    estimate_initial_box_vel_y,
    estimate_initial_box_vel_y_accurate,
    update_box_y_thermodynamic,
)
from graph_utils import prepare_traj
from itpo_weights import DatasetType
from simulator_model import Model as VelocityModel
from training_utils import GNNModel, ModelInputs, freeze_normalizer, huber_loss
from utils import (
    build_velocity_graph_correction,
    calc_p_ratio_box_tensor,
    get_correct_edge_attr,
    load_and_split_dataset,
)


### Load Data

In [ ]:
poisson_buckets = [
    {"max": 0.1, "count": 100},                # P < 0.1
    {"min": 0.1, "max": 0.2, "count": 100},    # 0.1 <= P < 0.2
    {"min": 0.2, "count": 200}                 # P >= 0.2
]

dataset_type = DatasetType.NodeOptimized

train_files, val_files, test_files = load_and_split_dataset(
    registry_path="./data/data_registry.csv",
    target_data_type=dataset_type,
    possion_buckets=poisson_buckets,
    split_ratios=(0.7, 0.15, 0.15),
    seed=42
)

# Load actual data
data = {
    'train': {},
    'val' : {},
    'test' : {},
}

print("Loading data...")
for key in data.keys():
    if key == 'train':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(train_files, desc=f"{key:<5} data")]
    elif key == 'val':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(val_files, desc=f"{key:<5} data")]
    elif key == 'test':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(test_files, desc=f"{key:<5} data")]
    else:
        raise ValueError(f"Unexpected key in data dictionary: {key}. ")

print("\nPreparing data...")
for data_type, sims in data.items():
    prepared = []
    for sim in tqdm(sims, desc=f"{data_type:<5} data"):
        prepared_sim = prepare_traj(sim, calc_angles=False)
        prepared.append(prepared_sim)
    data[data_type] = prepared

print(f"\nTrain data: {len(data['train'])} sims.")
print(f"Val data:   {len(data['val'])} sims.")
print(f"Test data:  {len(data['test'])} sims.")


#### Show $\nu$ distribution

In [ ]:
def visualize_nu_disribution(data: Dict):
    ps = {}
    all_values = []

    for data_type, sims in data.items():
        ps[data_type] = [calc_p_ratio_box_tensor(sim).item() for sim in sims]
        all_values.extend(ps[data_type])

    min_val = min(all_values)
    max_val = max(all_values)
    common_bins = np.linspace(min_val, max_val, 30) 

    for data_type, values in ps.items():
        plt.hist(
        values, 
        bins=common_bins, 
        edgecolor='black', 
        alpha=0.6, 
        label=f"{data_type} data"
    )

    plt.title("$\\nu$ distribution")
    plt.xlabel("GT LAMMPS $\\nu$")
    plt.ylabel("N")
    plt.legend()
    plt.show()

visualize_nu_disribution(data)


### Model training

#### Define training and refinement functions

In [ ]:
def train_h0_model(
    data: List[List[Data]],
    gnn_simulator: GNNModel,
    epochs: int,
    learning_rate: float,
    gamma: float,
    train_limit: int,
    accumulation_steps: int,
    freeze_norm_epoch: int,
    device: str = 'cuda',
    model_save_directory: Optional[str] = None,
) -> GNNModel:

    if model_save_directory is None:
        model_save_directory = os.path.join("./trained_models", f"{dataset_type}", "cascade", "h0")

    if not os.path.exists(model_save_directory):
        os.makedirs(model_save_directory, exist_ok=True)

    gnn_simulator.train()
    params = filter(lambda p: p.requires_grad, gnn_simulator.parameters())
    optimizer = torch.optim.Adam(params, lr=learning_rate, weight_decay=0.0)
    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma, last_epoch=-1)

    optimizer.zero_grad()
    for epoch in range(epochs):
        t_start = time.perf_counter()

        if epoch == freeze_norm_epoch:
            gnn_simulator.node_normalizer.frozen = True
            gnn_simulator.edge_normalizer.frozen = True
            gnn_simulator.output_normalizer.frozen = True

        # Trackers
        total_acc_loss = 0
        train_samples = 0

        gnn_simulator.train()
        for sim in data:
            random_starting_points = [i for i in range(train_limit)]

            for i, idx in enumerate(random_starting_points):
                indices = [step + idx for step in range(1)]
                target_idx = 1 + idx

                # Clone and Detach to prevent memory leaks from previous graphs
                input_graphs_raw = [sim[k].detach().cpu() for k in indices]

                # Construct input graph
                input_graph = build_velocity_graph_correction(input_graphs_raw, panic_at_positions=False).to(device) # default

                # Construct ModelInputs
                model_inputs = ModelInputs(
                    input_graphs_raw[-1].to(device),
                    input_graphs_raw[-1].to(device),
                    sim[target_idx].to(device)
                )

                # Forward & Loss
                model_output = gnn_simulator(input_graph, is_training=True)
                acc_loss = huber_loss(gnn_simulator, model_output, model_inputs, is_training=True)

                # Backward
                loss_for_backward = acc_loss / accumulation_steps
                loss_for_backward.backward()

                # Optimization
                if (i + 1) % accumulation_steps == 0:
                    torch.nn.utils.clip_grad_norm_(gnn_simulator.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()

                total_acc_loss += acc_loss.item()
                train_samples += 1

            optimizer.step()
            optimizer.zero_grad()

        lr_scheduler.step()

        avg_train_loss = total_acc_loss / train_samples

        if epoch % 1 == 0:
            gnn_simulator.save_checkpoint(os.path.join(model_save_directory, f"checkpoint_epoch_{epoch}.pt"))

        t_stop = time.perf_counter()
        print(
            f"Epoch {epoch + 1:>3} | "
            f"Train Loss: {avg_train_loss:.3e} | "
            f"Time: {t_stop - t_start:.2f} s"
        )

    return gnn_simulator


def train_cascade(
    models: List[GNNModel],
    ground_truth_sequence: List[Data], 
    optimizer: torch.optim.Adam,
    accumulation_steps: int, 
    global_step_idx: int,
    barostat_config: dict,
    train_limit: int = 20,
    device: str = "cuda"
) -> Tuple[float, float]:

    target_model_idx = len(models) - 1 # trained model is always last

    num_particles = ground_truth_sequence[0].num_nodes
    C_coupling = barostat_config["C_coupling"]
    damping_coeff = barostat_config["damping"]
    target_pressure = barostat_config["target_pressure"]
    temperature = barostat_config["temperature"]
    dt = barostat_config["dt"]
    dump_period = barostat_config["default_skip"]

    if dump_period is None:
        # Using data with variable dump period
        sim_strain = (sim[1].box.x - sim[-1].box.x) / sim[0].box.x
        assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
        dump_period = int(assumed_rollout_length / len(sim)) + 1
    
    stride_dt = dump_period * dt
    W_y = C_coupling * num_particles * (stride_dt**2)
    damping_params = damping_coeff * num_particles * stride_dt

    r0 = ground_truth_sequence[0].edge_attr[:, -2]

    # Freeze weights of every model except last one
    for i, m in enumerate(models):
        if i == target_model_idx:
            m.train()
            for param in m.parameters():
                param.requires_grad = True
        else:
            m.eval()
            for param in m.parameters():
                param.requires_grad = False

    total_loss = 0
    samples_processed = 0

    # Main loop
    max_start_idx = len(ground_truth_sequence) - (target_model_idx + 2)
    valid_train_limit = min(train_limit, max_start_idx + 1)

    starting_indices = range(valid_train_limit) 

    for step_offset, start_idx in enumerate(starting_indices):
        
        # Extract the Window for this specific step
        # Window needed: [start, start + target_idx + 2]
        window_end = start_idx + target_model_idx + 2
        sequence_slice = ground_truth_sequence[start_idx : window_end]
        current_trajectory = [sequence_slice[0].to(device)]
        
        if start_idx == 0:
            current_box_vel_y = 0.0
        elif start_idx == 1:
            current_box_vel_y = estimate_initial_box_vel_y(
                ground_truth_sequence[start_idx - 1],
                ground_truth_sequence[start_idx],
                stride_dt
            )
        else:
            current_box_vel_y = estimate_initial_box_vel_y_accurate(
                ground_truth_sequence[start_idx - 2],
                ground_truth_sequence[start_idx - 1],
                ground_truth_sequence[start_idx],
                stride_dt,
            )

        # Frozen Model Prediction
        with torch.no_grad():
            for i in range(target_model_idx):            
                
                # Construct input graph
                input_graph = build_velocity_graph_correction(current_trajectory, panic_at_positions=False).to(device)

                # Construct ModelInputs
                model_inputs = ModelInputs(
                    current_trajectory[-2] if len(current_trajectory) > 1 else current_trajectory[-1],
                    current_trajectory[-1],
                    sequence_slice[i+1].to(device)
                )
                
                # Forward
                pred_delta = models[i](input_graph, is_training=False)

                # Update next state
                next_step_pred = models[i].update(model_inputs, pred_delta, recalc_edges=False)

                # Apply Barostat
                b0: Tensor = model_inputs.cur_graph.box_tensor[0]
                b1: Tensor = model_inputs.target_graph.box_tensor[0]
                box_compression_factor = b1 / b0

                new_box_tensor: Tensor = next_step_pred.box_tensor.clone()
                
                # Scale Lx (Box Compression)
                new_box_tensor[0] = new_box_tensor[0] * box_compression_factor
                
                # Update Ly
                new_ly, new_vel_y = update_box_y_thermodynamic(
                    positions=next_step_pred.pos,
                    edge_index=model_inputs.cur_graph.edge_index,
                    edge_attr=model_inputs.cur_graph.edge_attr,
                    current_box=model_inputs.cur_graph.box_tensor,
                    r0=r0,
                    box_vel_y=current_box_vel_y,
                    W_y=W_y,
                    damping=damping_params,
                    stride_dt=stride_dt,
                    target_pressure=target_pressure,
                    temperature=temperature,
                )
                new_box_tensor[1] = new_ly
                current_box_vel_y = new_vel_y
                
                # Assign new box to the predicted state
                next_step_pred.box_tensor = new_box_tensor

                # Recalculate edges
                next_step_pred.edge = get_correct_edge_attr(
                    next_step_pred,
                    recompute_stiff=False,
                    panic_at_nontensor_box=True,
                )

                # Append to history (Detach to stop gradients)
                current_trajectory.append(next_step_pred.detach())


        # Construct target input graph
        target_input_graph = build_velocity_graph_correction(current_trajectory, panic_at_positions=False).to(device)
        gt_target = sequence_slice[target_model_idx + 1].to(device)
        
        # Construct target ModelInputs
        target_inputs = ModelInputs(
            current_trajectory[-2],
            current_trajectory[-1],
            gt_target
        )

        # Forward and Loss
        prediction = models[target_model_idx](target_input_graph, is_training=True)
        loss = huber_loss(models[target_model_idx], prediction, target_inputs, is_training=True)
        
        # Backward
        loss_backward = loss / accumulation_steps
        loss_backward.backward()

        total_loss += loss.item()
        samples_processed += 1

        # Optimization 
        current_global_step = global_step_idx + step_offset
        if (current_global_step + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(models[target_model_idx].parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

    return total_loss / samples_processed, samples_processed


def train_h_model(
    data: List[List[Data]],
    models: List[GNNModel],
    epochs: int,
    learning_rate: float,
    barostat_config: dict,
    train_limit: int,
    accumulation_steps: int,
    freeze_norm_epoch: int,
    device: str = 'cuda',
    model_save_directory: Optional[str] = None,
) -> GNNModel:

    if model_save_directory is None:
        model_save_directory = os.path.join("./trained_models", f"{dataset_type}", "cascade", f"h{len(models)-1}")

    if not os.path.exists(model_save_directory):
        os.makedirs(model_save_directory, exist_ok=True)

    # Trained model is always the last in sequence
    params = filter(lambda p: p.requires_grad, models[-1].parameters())
    optimizer = torch.optim.Adam(params, lr=learning_rate, weight_decay=0.0)
    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, 0.995, last_epoch=-1)
    optimizer.zero_grad()

    step_counter = 0

    for epoch in range(epochs):
        t_start = time.time()

        if epoch == freeze_norm_epoch:
            models[-1].node_normalizer.frozen = True
            models[-1].edge_normalizer.frozen = True
            models[-1].output_normalizer.frozen = True

        # Trackers
        total_acc_loss = 0
        train_samples = 0
        
        for sim in data:

            # Call the train function
            training_loss, samples_processed = train_cascade(
                models=models,
                ground_truth_sequence=sim,
                optimizer=optimizer,
                accumulation_steps=accumulation_steps,
                global_step_idx=step_counter,
                barostat_config=barostat_config,
                train_limit=train_limit,
                device=device
            )
            
            total_acc_loss += training_loss
            step_counter += samples_processed
            train_samples += 1

        # Remaining gradients 
        if step_counter % accumulation_steps != 0:
            optimizer.step()
            optimizer.zero_grad()

        lr_scheduler.step()
        
        # Statistics
        avg_train_loss = total_acc_loss / train_samples

        # Save model
        if epoch % 1 == 0:
            models[-1].save_checkpoint(os.path.join(model_save_directory, f"checkpoint_epoch_{epoch}.pt"))

        t_stop = time.time()
        print(
            f"Epoch {epoch + 1:>3} | "
            f"Train Loss: {avg_train_loss:.3e} | "
            f"Time: {t_stop - t_start:.2f} s"
        )

    return models[-1]


def refine_cascade(
    data: List[List[Data]],
    models: List[VelocityModel],
    epochs: int,
    barostat_config: dict,
    train_limit: int = 10,
    rollout_steps: int = 5,
    freeze_norm_epoch: int = 0,
    learning_rate: float = 5e-5,
    gamma: float = 0.995,
    device: str = "cuda",
) -> List[VelocityModel]:
    
    # Estimate box compression factor from GT
    b0 = data[0][2].box_tensor[0]
    b1 = data[0][3].box_tensor[0]
    box_compression_factor = b1 / b0

    stride_dt = barostat_config["default_skip"] * barostat_config["dt"]

    # Unfreeze all models and collect parameters
    all_params = []
    for m in models:
        m.train()
        for param in m.parameters():
            param.requires_grad = True
        all_params.append(m.parameters())

    # Chain parameters for a single optimizer
    optimizer = torch.optim.Adam(chain(*all_params), lr=learning_rate, weight_decay=0.0)
    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma, last_epoch=-1)

    for epoch in range(epochs):
        t_start = time.perf_counter()

        # Trackers
        total_per_step_loss = {h : 0 for h in range(len(models))}
        total_acc_loss = 0
        train_samples = 0

        # Freeze normalizers to stabilize fine-tuning (in case they aren't frozen already)
        if epoch == freeze_norm_epoch:
            for m in models:
                m.node_normalizer.frozen = True
                m.edge_normalizer.frozen = True
                m.output_normalizer.frozen = True

        for sim in data:

            r0 = sim[0].edge_attr[:, -2]

            # Random starting points
            random_starting_points = [i for i in range(train_limit)]

            for start_idx in random_starting_points:
                
                # Stop if there isn't enough graphs left in the simulation
                if start_idx + rollout_steps + len(models) >= len(sim):
                    continue

                optimizer.zero_grad()
                
                # Initialization
                # Randomly pick how much GT history to start with.
                init_hist_len = random.randint(1, len(models)+1)
                
                # Load ground truth frames
                # indices: [start_idx, start_idx+1, ... start_idx + init_hist_len - 1]
                indices = [start_idx + k for k in range(init_hist_len)]
                
                # Detach initial inputs
                current_window_graphs = [sim[k].detach().to(device) for k in indices]

                # Barostat Initialization
                if len(current_window_graphs) >= 3:
                    current_box_vel_y = estimate_initial_box_vel_y_accurate(
                        current_window_graphs[-3],
                        current_window_graphs[-2],
                        current_window_graphs[-1],
                        stride_dt
                    )
                elif len(current_window_graphs) == 2:
                    current_box_vel_y = estimate_initial_box_vel_y(
                        current_window_graphs[-2],
                        current_window_graphs[-1],
                        stride_dt
                    )
                else:
                    current_box_vel_y = 0.0

                # Rollout
                rollout_loss = 0
                per_step_loss  = {h : 0 for h in range(len(models))}
                for step in range(rollout_steps):

                    # Select the correct model from the cascade
                    # history len 1 -> use models[0] (h0)
                    # history len 2 -> use models[1] (h1)
                    # history len N -> use models[N-1]
                    history_len = len(current_window_graphs)
                    model_idx = min(history_len - 1, len(models) - 1)
                    active_model = models[model_idx]
                    
                    # Prepare Inputs
                    # NOT detaching here. Gradients must flow.
                    input_graph = build_velocity_graph_correction(current_window_graphs[-(len(models))::], panic_at_positions=False).to(device)
                    
                    target_idx = start_idx + history_len 
                    target_graph = sim[target_idx].to(device)

                    # Init ModelInputs
                    model_inputs = ModelInputs(
                        current_window_graphs[-2] if history_len > 1 else current_window_graphs[-1],
                        current_window_graphs[-1],
                        target_graph
                    )

                    # Forward and Loss
                    model_output = active_model(input_graph, is_training=True)
                    step_loss = huber_loss(active_model, model_output, model_inputs, is_training=True)
                    rollout_loss += step_loss
                    per_step_loss[model_idx] += step_loss

                    # Update next state
                    pred_graph_next = active_model.update(model_inputs, model_output, recalc_edges=True)

                    # Update box Lx
                    new_lx = pred_graph_next.box_tensor[0] * box_compression_factor
                    
                    # Apply barostat for Ly
                    W_y = barostat_config["C_coupling"] * pred_graph_next.num_nodes * (stride_dt ** 2)
                    damping = barostat_config["damping"] * pred_graph_next.num_nodes * stride_dt
                    new_ly, new_vel_y = update_box_y_thermodynamic(
                        positions=pred_graph_next.pos,
                        edge_index=model_inputs.cur_graph.edge_index,
                        edge_attr=model_inputs.cur_graph.edge_attr,
                        current_box=model_inputs.cur_graph.box_tensor,
                        r0=r0.to(pred_graph_next.pos.device),
                        box_vel_y=current_box_vel_y, 
                        W_y=W_y,
                        damping=damping,
                        stride_dt=stride_dt,
                        target_pressure=barostat_config["target_pressure"],
                        temperature=barostat_config["temperature"],
                    )
                    new_box_tensor = torch.stack([new_lx, new_ly])
                    current_box_vel_y = new_vel_y

                    # Assign new box 
                    pred_graph_next.box_tensor = new_box_tensor
                    
                    # Update edge attr
                    pred_graph_next.edge_attr = get_correct_edge_attr(pred_graph_next, recompute_stiff=False, panic_at_nontensor_box=True)

                    # Append
                    current_window_graphs.append(pred_graph_next)

                # Average loss over the rollout steps
                final_loss = rollout_loss / rollout_steps
                final_loss.backward()

                # Clip gradients
                torch.nn.utils.clip_grad_norm_(chain(*all_params), max_norm=1.0)
                
                optimizer.step()

                total_acc_loss += final_loss.item()
                for h, l in per_step_loss.items():
                    total_per_step_loss[h] = total_per_step_loss[h] + l
                train_samples += 1

        total_acc_loss /= train_samples
        for h, l in total_per_step_loss.items():
            total_per_step_loss[h] = total_per_step_loss[h] / train_samples
        lr_scheduler.step()
        
        t_stop = time.perf_counter()
        per_model_section = " | ".join([f"h{h}: {l:.4e}" for h, l in total_per_step_loss.items()])
        print(f"Epoch {epoch:<3} | "
              f"fine-tuning loss: {total_acc_loss:.4e} | "
              f"{per_model_section} | "
              f"Time: {t_stop - t_start:.2f} s.")

    return models


#### Independent one-by-one training

In [ ]:
device = "cuda"

max_history = 4
epochs = 10
freeze_norm_epoch = 5
train_sims = 100
train_limit = 15
accumulation_steps = 10
learning_rate = 1e-3
gamma = 0.995

mp_layers = 2
mlp = 3
hidden_size = 128

for current_history in range(max_history+1):
    
    # Model output path
    model_save_directory = os.path.join("./trained_models", f"{dataset_type}", "cascade", f"h{current_history}")
    if not os.path.exists(model_save_directory):
        os.makedirs(model_save_directory, exist_ok=True)

    print(f"Starting history {current_history} model training...")
    
    if current_history == 0:
        # Init fresh model
        init_graph = build_velocity_graph_correction([data['train'][0][i].cpu().detach() for i in range(current_history + 1)], panic_at_positions=False).to(device)
        gnn_simulator = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)

        trained_h0_model = train_h0_model(
            data=data['train'],
            gnn_simulator=gnn_simulator,
            epochs=epochs,
            learning_rate=learning_rate,
            gamma=gamma,
            train_limit=train_limit,
            accumulation_steps=accumulation_steps,
            freeze_norm_epoch=freeze_norm_epoch,
            device=device,
            model_save_directory=model_save_directory
        )

        h0_save_path = os.path.join(model_save_directory, f"model_h{current_history}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt")
        trained_h0_model.save_checkpoint(h0_save_path)
        print(f"History 0 model trained and saved to {h0_save_path}.")
    
    else:
        
        models = []        
        
        # Load and freeze models up to current history, then add a fresh one
        for past_history in range(current_history):
            
            # Construct input graph
            init_graph = build_velocity_graph_correction(
                [data['train'][0][i].cpu().detach() for i in range(past_history+1)],
                panic_at_positions=False
            ).to(device)
            
            # Initialize past h model
            past_history_model = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
            
            # Load checkpoint
            past_model_path = os.path.join("./trained_models", f"{dataset_type}", "cascade", f"h{past_history}", f"model_h{past_history}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt")
            past_history_model.load_checkpoint(past_model_path)
            
            # Freeze normalizer and model weights
            past_history_model = freeze_normalizer(past_history_model)
            for param in past_history_model.parameters():
                param.requires_grad = False
            
            # Append 
            models.append(past_history_model)

        # Init fresh model for training
        init_graph = build_velocity_graph_correction([data['train'][0][i].cpu().detach() for i in range(current_history + 1)], panic_at_positions=False).to(device)
        gnn_simulator = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        
        # Init Adam optimizer and LR scheduler 
        optimizer = torch.optim.Adam(gnn_simulator.parameters(), lr=learning_rate, weight_decay=0.0)
        lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma, last_epoch=-1)
        
        models.append(gnn_simulator)

        if str(dataset_type) == "node_optimized":
            barostat_config = barostat_parameters.node_optimizated
        elif str(dataset_type) == "stiff_optimized":
            barostat_config = barostat_parameters.stiff_optimized
        else:
            raise ValueError('Unrecognized dataset type.')

        trained_current_history_model = train_h_model(
            data=data['train'],
            models=models,
            epochs=epochs,
            learning_rate=learning_rate,
            barostat_config=barostat_config,
            train_limit=train_limit,
            accumulation_steps=accumulation_steps,
            freeze_norm_epoch=freeze_norm_epoch,
            device=device,
        )

        h_save_path = os.path.join(model_save_directory, f"model_h{current_history}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt")
        trained_current_history_model.save_checkpoint(h_save_path)
        print(f"History {current_history} model trained and saved to {h_save_path}.")
    

#### Refinement 

In [ ]:
epochs = 10
freeze_norm_epoch = 5
train_sims = 100
train_limit = 15
rollout_steps = 10
accumulation_steps = 10
learning_rate = 1e-3
gamma = 0.995

mp_layers = 2
mlp = 3
hidden_size = 128

if str(dataset_type) == "node_optimized":
    barostat_config = barostat_parameters.node_optimizated
elif str(dataset_type) == "stiff_optimized":
    barostat_parameters.stiff_optimized
else:
    raise ValueError('Unrecognized dataset type.')

finetuned_models = refine_cascade(
    data=data['train'][:25]+data['ft'][:25]+data['val'][:25]+data['test'][:25],
    models=models,
    epochs=epochs,
    barostat_config=barostat_config,
    train_limit=train_limit,
    rollout_steps=rollout_steps,
    freeze_norm_epoch=freeze_norm_epoch,
    learning_rate=learning_rate,
    gamma=gamma,
    device=device
)

model_save_path = os.path.join("./trained_models", f"{dataset_type}", "cascade", "refined")
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path, exist_ok=True)

for h, m in enumerate(finetuned_models):
    m.save_checkpoint(os.path.join(model_save_path, f"model_refined_h{h}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt"))